# LexRAG on Kaggle (modified)

Benchmark RAG untuk konsultasi hukum multi-turn (data hukum Tiongkok) — pipeline: **Retrieval (BM25 + Dense) → Generation → Evaluation (metric + LLM-as-a-Judge)**.

**Sebelum Run All:**
1. Settings → GPU → **T4** (embedding 17.229 artikel hukum)
2. Settings → **Internet ON** (clone repo + download model HF)
3. Isi `GROQ_API_KEY` di cell Konfigurasi (gratis: https://console.groq.com → API Keys)

**Estimasi pemakaian LLM (Groq free tier: 1000 request/hari, 30 RPM):**
- Generate: `N_DIALOGUES × 5` call
- LLM-as-a-Judge: `N_DIALOGUES × 5` call (jika `RUN_JUDGE=True`)
- Default `N_DIALOGUES=20` → ±100+100 call = 20% kuota harian

Modifikasi terhadap repo asli: fix NameError saat jumlah sampel kelipatan `batch_size`, fix KeyError config judge, fix hasil evaluasi generation yang terfilter kosong, normalisasi embedding (cosine untuk FAISS inner-product), import berat dijadikan lazy (pyserini/zhipuai/modelscope tidak wajib), backoff pada judge, dan agregasi skor judge.

In [ ]:
!pip install -q sentence-transformers faiss-cpu bm25s jieba langid rouge-score bert-score nltk scikit-learn openai

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/irnakhalda/LEXRAG_TEST.git"
if not os.path.exists("LEXRAG_TEST"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir("LEXRAG_TEST")
sys.path.insert(0, os.path.abspath("src"))
print("Working dir:", os.getcwd())

## Konfigurasi
Isi API key Groq di bawah ini, lalu atur parameter eksperimen.

In [ ]:
import json, os, shutil

GROQ_API_KEY = "gsk_GANTI_DENGAN_KEY_ANDA"
assert not GROQ_API_KEY.startswith("gsk_GANTI"), "Isi dulu GROQ_API_KEY di cell ini!"

LLM_CONFIG = {
    "model_type": "openai",          # memakai OpenAI-compatible endpoint (Groq)
    "model_name": "qwen/qwen3.8-27b", # model gratis di Groq
    "api_base": "https://api.groq.com/openai/v1",
    "api_key": GROQ_API_KEY,
    "max_retries": 6,
    "max_parallel": 3,               # jaga di bawah 30 RPM (free tier)
}

N_DIALOGUES = 20     # 20 dialogues x 5 turn = 100 panggilan LLM per tahap
TOP_N = 5            # jumlah artikel hukum yang disuntikkan ke prompt
RUN_DENSE = True     # True: embedding 17.229 artikel dengan BGE-base-zh (±2-5 mnt di T4)
RUN_JUDGE = True     # LLM-as-a-Judge (+N_DIALOGUES*5 panggilan LLM)
RUN_BERTSCORE = False # True: tambah BERTScore (download model ~400MB)

RETRIEVAL_METRICS = ["recall", "precision", "f1", "mrr", "ndcg"]
K_VALUES = [1, 3, 5, 10]
print(f"LLM: {LLM_CONFIG['model_name']} @ {LLM_CONFIG['api_base']}")
print(f"Subset: {N_DIALOGUES} dialogues ({N_DIALOGUES * 5} generation calls)")

In [ ]:
# Cek cepat: key & endpoint valid sebelum pipeline panjang
from openai import OpenAI
client = OpenAI(api_key=LLM_CONFIG["api_key"], base_url=LLM_CONFIG["api_base"])
pong = client.chat.completions.create(
    model=LLM_CONFIG["model_name"],
    messages=[{"role": "user", "content": "balas dengan satu kata: OK"}],
    temperature=0.0,
)
print("API OK ->", pong.choices[0].message.content)

## 1. Data prep (subset)
Dataset penuh = 1.013 dialogues x 5 turn = 5.065 panggilan LLM (melebihi kuota free tier), jadi kita ambil subset. File query memakai hasil rewrite yang sudah di-precompute (tanpa LLM tambahan). Output run sebelumnya dibersihkan agar tidak duplikat.

In [ ]:
for p in ["data/generated_responses.jsonl", "data/retrieval", "data/generated_samples",
          "data/prompt", "data/results", "data/retrieval/report.jsonl"]:
    if os.path.isdir(p):
        shutil.rmtree(p)
    elif os.path.exists(p):
        os.remove(p)

dataset = json.load(open("data/dataset.json", encoding="utf-8"))
subset = dataset[:N_DIALOGUES]
subset_ids = {d["id"] for d in subset}

os.makedirs("data/subset", exist_ok=True)
json.dump(subset, open("data/subset/dataset.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)

n_match = 0
with open("data/samples/rewrite_question.jsonl", encoding="utf-8") as fin, \
     open("data/subset/rewrite_question.jsonl", "w", encoding="utf-8") as fout:
    for line in fin:
        if json.loads(line)["id"] in subset_ids:
            fout.write(line)
            n_match += 1

n_laws = sum(1 for _ in open("data/law_library.jsonl", encoding="utf-8"))
print(f"Subset dialogues : {len(subset)} ({len(subset) * 5} turns)")
print(f"Query file       : {n_match} dialogues matched")
print(f"Corpus           : {n_laws} articles")
assert n_match == len(subset), "rewrite_question sample tidak lengkap untuk subset ini!"

## 2a. Retrieval — Sparse (BM25 via bm25s)
Tokenisasi Chinese dengan jieba. Tidak butuh Java (pyserini dimatikan via lazy import).

In [ ]:
from pipeline import RetrieverPipeline, EvaluatorPipeline

bm25_res_path = "data/retrieval/res/retrieval_bm25_bm25s.jsonl"

rp = RetrieverPipeline()
rp.run_retriever(model_type="bm25", bm25_backend="bm25s",
                 question_file_path="data/subset/rewrite_question.jsonl",
                 law_path="data/law_library.jsonl")

report_bm25 = EvaluatorPipeline().run_evaluator(
    eval_type="retrieval",
    results_path=bm25_res_path,
    metrics=RETRIEVAL_METRICS,
    k_values=K_VALUES,
)

## 2b. Retrieval — Dense (BGE-base-zh + FAISS)
Embedding lokal (gratis) di GPU, index inner-product pada vektor yang sudah dinormalisasi (= cosine).

In [ ]:
report_dense = None
dense_res_path = "data/retrieval/res/retrieval_BGE-base-zh.jsonl"

if RUN_DENSE:
    import torch
    print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")
    rp = RetrieverPipeline()
    rp.run_retriever(model_type="BGE-base-zh", faiss_type="FlatIP",
                     question_file_path="data/subset/rewrite_question.jsonl",
                     law_path="data/law_library.jsonl")
    report_dense = EvaluatorPipeline().run_evaluator(
        eval_type="retrieval",
        results_path=dense_res_path,
        metrics=RETRIEVAL_METRICS,
        k_values=K_VALUES,
    )

In [ ]:
import pandas as pd

rows = {}
if report_bm25:
    rows["BM25 (bm25s)"] = {k: v for k, v in report_bm25.items() if k != "results_path"}
if report_dense:
    rows["Dense (BGE-base-zh)"] = {k: v for k, v in report_dense.items() if k != "results_path"}
pd.DataFrame(rows).T

## 3. Generation — Groq `qwen/qwen3.8-27b`
Prompt bawaan `LegalPromptBuilder` (system berisi top-N artikel hukum + riwayat percakapan). Hasil ditulis ke `data/generated_responses.jsonl`.

In [ ]:
from pipeline import GeneratorPipeline

retrieval_path = dense_res_path if RUN_DENSE else bm25_res_path
assert os.path.exists(retrieval_path), f"Belum ada hasil retrieval: {retrieval_path}"

if os.path.exists("data/generated_responses.jsonl"):
    os.remove("data/generated_responses.jsonl")

gen_pipeline = GeneratorPipeline(model_type="openai", config=LLM_CONFIG)
gen_pipeline.run_generator(
    raw_data_path="data/subset/dataset.json",
    retrieval_data_path=retrieval_path,
    top_n=TOP_N,
    max_retries=LLM_CONFIG["max_retries"],
    max_parallel=LLM_CONFIG["max_parallel"],
    batch_size=5,
)

n_resp = sum(1 for _ in open("data/generated_responses.jsonl", encoding="utf-8"))
print(f"Total respons: {n_resp} (expected {N_DIALOGUES * 5})")
assert n_resp == N_DIALOGUES * 5, "Sebagian respons gagal — coba run ulang cell ini (retry aman setelah file dihapus)."

## 4. Evaluasi generation
ROUGE-L, BLEU 1-4, METEOR, keyword accuracy, char-F1 (±BERTScore opsional) per turn.

In [ ]:
generation_metrics = ["rouge", "bleu", "keyword_accuracy", "char-scores", "meteor"]
if RUN_BERTSCORE:
    generation_metrics.append("bert-score")

gen_report = EvaluatorPipeline().run_evaluator(
    eval_type="generation",
    metrics=generation_metrics,
    data_path="data/subset/dataset.json",
    response_file="data/generated_responses.jsonl",
)

pd.DataFrame(gen_report).T.rename_axis("turn")

## 5. LLM-as-a-Judge (opsional)
5 dimensi (准确性 / 满足用户需求 / 清晰度 / 逻辑连贯性 / 完备性) + skor komprehensif 1-10, dinilai oleh LLM yang sama.

In [ ]:
from eval.evaluator import LLMJudge
from eval.llm_as_judge.run_judge import aggregate_judge_results

if RUN_JUDGE:
    judge = LLMJudge(LLM_CONFIG)
    judge.evaluate(
        data_path="data/subset/dataset.json",
        gen_path="data/generated_responses.jsonl",
    )
    judge_summary = aggregate_judge_results()
    print(json.dumps(judge_summary, indent=2, ensure_ascii=False))
else:
    print("RUN_JUDGE=False — dilewati.")

## 6. Contoh kualitatif

In [ ]:
ret_map, resp_map = {}, {}
with open(retrieval_path, encoding="utf-8") as f:
    for line in f:
        e = json.loads(line)
        ret_map[e["id"]] = e
with open("data/generated_responses.jsonl", encoding="utf-8") as f:
    for line in f:
        e = json.loads(line)
        resp_map[e["id"]] = e

for turn in [1, 2]:
    ex_id = f"{subset[0]['id']}_turn{turn}"
    conv = subset[0]["conversation"][turn - 1]
    recall = ret_map[subset[0]["id"]]["conversation"][turn - 1]["question"]["recall"]
    top_articles = sorted(recall, key=lambda x: x["score"], reverse=True)[:TOP_N]
    print("=" * 100)
    print(f"[{ex_id}] Q : {conv['user']}")
    print(f"- Referensi  : {conv['assistant'][:300]}")
    print(f"- Generated  : {resp_map.get(ex_id, {}).get('response', '')[:300]}")
    print(f"- Gold articles: {conv['article']}")
    print("- Top retrieved:")
    for r in top_articles:
        print(f"    {r['score']:.4f}  {r['article']['name']}")

### Catatan
- **Ganti subset**: naikkan `N_DIALOGUES` (mis. 50 → 250+250 call) — pantau kuota di https://console.groq.com/usage. Jika kena `429`, pipeline otomatis retry dengan backoff.
- **Rerun**: cell data-prep membersihkan semua output lama; kalau hanya ingin mengulang generation, hapus `data/generated_responses.jsonl` dulu (file ditulis mode append).
- **Retriever lain**: ganti `retrieval_path` ke `data/retrieval/res/retrieval_bm25_bm25s.jsonl` untuk generate dengan BM25, atau set `RUN_DENSE=False`.
- **Jangan publish notebook** ini jika key Groq sudah tertanam — buat key baru bila key lama bocor.